In [ ]:
print("hello world")

In [ ]:
import os
import pandas as pd


In [ ]:
df = pd.read_csv('/home/jovyan/output/data/agentA/train.csv')

In [ ]:
# Print basic information about the dataset
print("Dataset Shape:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\n" + "="*50)

# Display first few rows
print("First 5 rows:")
print(df.head())
print("\n" + "="*50)

# Display basic info about the dataframe
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)

# Display statistical summary
print("Statistical Summary:")
print(df.describe())
print("\n" + "="*50)

# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*50)

# Display column names and data types
print("Columns and Data Types:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your dataframe is called 'df'
# First, let's prepare the data

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

def preprocess_data(df, encoders=None, scaler=None, imputers=None, is_training=True):
    """
    Comprehensive preprocessing for telecom customer data
    
    Parameters:
    df: Input dataframe
    encoders: Dictionary of fitted encoders (for test data)
    scaler: Fitted scaler (for test data)
    imputers: Dictionary of fitted imputers (for test data)
    is_training: Boolean indicating if this is training data
    
    Returns:
    X: Feature matrix
    y: Target variable (if training)
    feature_columns: List of feature column names
    fitted_objects: Dictionary containing fitted preprocessors
    """
    
    data = df.copy()
    
    # Separate features by type for better handling
    # Demographic features
    demographic_features = ['sex', 'age', 'innet_dura']
    
    # Usage and billing features
    usage_features = [
        'arpu', 'l3m_avg_mou', 'l3m_avg_dou', 'l3m_avg_bill_dura',
        'cm_tot_bill_dura', 'cm_local_voice_dura', 'cm_flux_use',
        'flux_4g_use', 'flux_up_4g_sum', 'flux_down_4g_sum'
    ]
    
    # Time-based usage features
    time_usage_features = [
        'wday_day_flux', 'wday_night_flux', 'nwday_day_flux', 'nwday_night_flux'
    ]
    
    # Plan and service features
    plan_features = [
        'cm_flux_tot_cnt', 'cm_base_plan_flux', 'cm_base_plan_flux_use',
        'cm_chos_plan_flux', 'cm_chos_plan_flux_use', 'cm_over_plan_flux',
        'gprs_days'
    ]
    
    # Binary indicator features
    binary_features = [
        'is_fam_vnet_user', 'is_ent_vnet_user', 'is_bd_status_abnormal',
        'is_10g_pon', 'is_bd_tv', 'if_nulim_prod'
    ]
    
    # Broadband features
    broadband_features = ['bd_flux_m', 'bd_dur_m', 'bd_cnt_m']
    
    # Activity features
    activity_features = [
        'user_duration_m', 'login_times_m', 'click_times_m', 'watch_times_m',
        'open_day_m', 'click_day_m'
    ]
    
    # Contract and group features
    contract_features = ['term_cont_mon', 'term_cont_dfee', 'grp_cnts']
    
    # Usage pattern features
    usage_pattern_features = ['out_gprs', 'out_call']
    
    # Content consumption features
    content_features = [
        'video_cnt_m', 'read_time_m', 'read_cnt_m', 'music_cnt_m',
        'caijing_time_m', 'caijing_cnt_m', 'travel_time_m', 'travel_cnt_m',
        'game_cnt_m', 'edu_time_m', 'edu_cnt_m', 'fashion_time_m'
    ]
    
    # Customer preference features
    preference_features = [
        'if_high_games_cust', 'if_like_games_cust', 'if_like_video_cust'
    ]
    
    # App usage features (with missing values)
    app_usage_features = [
        'gm_use_dur', 'gm_dayt_use_dur', 'gm_ngt_use_dur',
        'shrt_vid_use_dur', 'shrt_vid_dayt_use_dur', 'shrt_vid_ngt_use_dur',
        'long_vid_use_dur', 'long_vid_dayt_use_dur', 'long_vid_ngt_use_dur',
        'anchor_use_dur', 'anchor_dayt_use_dur', 'anchor_ngt_use_dur',
        'wtch_liv_use_dur', 'wtch_liv_dayt_use_dur', 'wtch_liv_ngt_use_dur',
        'netdisk_use_dur', 'netdisk_dayt_use_dur', 'netdisk_ngt_use_dur'
    ]
    
    # User label features (with missing values)
    user_label_features = [
        'hi_flux_usr_lbl', 'sev_vid_usr_lbl', 'liv_usr_lbl',
        'netdisk_usr_lbl', 'vid_usr_lbl', 'read_usr_lbl',
        'gm_usr_lbl', 'msc_usr_lbl'
    ]
    
    # Handle missing values
    if is_training:
        imputers = {}
        
        # For app usage features, use median imputation
        imputer_app = SimpleImputer(strategy='median')
        data[app_usage_features] = imputer_app.fit_transform(data[app_usage_features])
        imputers['app_usage'] = imputer_app
        
        # For user label features, use mode (most frequent) imputation
        imputer_labels = SimpleImputer(strategy='most_frequent')
        data[user_label_features] = imputer_labels.fit_transform(data[user_label_features])
        imputers['user_labels'] = imputer_labels
        
    else:
        # Apply fitted imputers
        data[app_usage_features] = imputers['app_usage'].transform(data[app_usage_features])
        data[user_label_features] = imputers['user_labels'].transform(data[user_label_features])
    
    # Create new engineered features
    print("Creating engineered features...")
    
    # Usage intensity features
    data['total_flux_usage'] = data['flux_4g_use'] + data['cm_flux_use']
    data['flux_efficiency'] = data['total_flux_usage'] / (data['arpu'] + 1)
    data['voice_data_ratio'] = data['l3m_avg_mou'] / (data['l3m_avg_dou'] + 1)
    
    # Time-based usage patterns
    data['day_night_flux_ratio'] = (data['wday_day_flux'] + data['nwday_day_flux']) / \
                                   (data['wday_night_flux'] + data['nwday_night_flux'] + 1)
    data['weekday_weekend_ratio'] = (data['wday_day_flux'] + data['wday_night_flux']) / \
                                    (data['nwday_day_flux'] + data['nwday_night_flux'] + 1)
    
    # Plan utilization
    data['base_plan_utilization'] = data['cm_base_plan_flux_use'] / (data['cm_base_plan_flux'] + 1)
    data['chosen_plan_utilization'] = data['cm_chos_plan_flux_use'] / (data['cm_chos_plan_flux'] + 1)
    data['over_plan_ratio'] = data['cm_over_plan_flux'] / (data['total_flux_usage'] + 1)
    
    # Activity intensity
    data['avg_daily_clicks'] = data['click_times_m'] / (data['open_day_m'] + 1)
    data['avg_session_duration'] = data['user_duration_m'] / (data['login_times_m'] + 1)
    data['engagement_score'] = data['watch_times_m'] / (data['click_times_m'] + 1)
    
    # Content consumption patterns
    data['total_content_time'] = data['read_time_m'] + data['caijing_time_m'] + \
                                data['travel_time_m'] + data['edu_time_m'] + data['fashion_time_m']
    data['total_content_count'] = data['video_cnt_m'] + data['read_cnt_m'] + data['music_cnt_m'] + \
                                 data['caijing_cnt_m'] + data['travel_cnt_m'] + data['game_cnt_m'] + data['edu_cnt_m']
    
    # App usage patterns (for users with app data)
    data['total_app_usage'] = data['gm_use_dur'] + data['shrt_vid_use_dur'] + \
                             data['long_vid_use_dur'] + data['anchor_use_dur'] + \
                             data['wtch_liv_use_dur'] + data['netdisk_use_dur']
    
    data['video_usage_total'] = data['shrt_vid_use_dur'] + data['long_vid_use_dur']
    data['day_night_app_ratio'] = (data['gm_dayt_use_dur'] + data['shrt_vid_dayt_use_dur'] + 
                                  data['long_vid_dayt_use_dur']) / \
                                 (data['gm_ngt_use_dur'] + data['shrt_vid_ngt_use_dur'] + 
                                  data['long_vid_ngt_use_dur'] + 1)
    
    # Customer value segments
    data['high_value_customer'] = ((data['arpu'] > data['arpu'].quantile(0.8)) & 
                                  (data['innet_dura'] > 12)).astype(int)
    data['heavy_data_user'] = (data['total_flux_usage'] > data['total_flux_usage'].quantile(0.8)).astype(int)
    data['multi_service_user'] = (data['is_fam_vnet_user'] + data['is_ent_vnet_user'] + 
                                 data['is_bd_tv'] + data['if_nulim_prod'] >= 2).astype(int)
    
    # Age groups
    data['age_group'] = pd.cut(data['age'], 
                              bins=[0, 25, 35, 45, 55, 100], 
                              labels=['young', 'adult', 'middle', 'senior', 'elderly'])
    
    # Tenure groups
    data['tenure_group'] = pd.cut(data['innet_dura'], 
                                 bins=[0, 6, 12, 24, 60, 1000], 
                                 labels=['new', 'short', 'medium', 'long', 'very_long'])
    
    # ARPU groups
    data['arpu_group'] = pd.cut(data['arpu'], 
                               bins=[0, 50, 100, 200, 1000], 
                               labels=['low', 'medium', 'high', 'premium'])
    
    # Encode categorical features
    categorical_features = ['age_group', 'tenure_group', 'arpu_group']
    
    if is_training:
        encoders = {}
        for feature in categorical_features:
            le = LabelEncoder()
            data[f'{feature}_encoded'] = le.fit_transform(data[feature].astype(str))
            encoders[feature] = le
    else:
        for feature in categorical_features:
            # Handle unseen categories
            known_classes = set(encoders[feature].classes_)
            data[feature] = data[feature].astype(str)
            data[feature] = data[feature].apply(lambda x: x if x in known_classes else encoders[feature].classes_[0])
            data[f'{feature}_encoded'] = encoders[feature].transform(data[feature])
    
    # Select final features for modeling
    base_features = (demographic_features + usage_features + time_usage_features + 
                    plan_features + binary_features + broadband_features + 
                    activity_features + contract_features + usage_pattern_features + 
                    content_features + preference_features + app_usage_features + 
                    user_label_features)
    
    engineered_features = [
        'total_flux_usage', 'flux_efficiency', 'voice_data_ratio',
        'day_night_flux_ratio', 'weekday_weekend_ratio',
        'base_plan_utilization', 'chosen_plan_utilization', 'over_plan_ratio',
        'avg_daily_clicks', 'avg_session_duration', 'engagement_score',
        'total_content_time', 'total_content_count',
        'total_app_usage', 'video_usage_total', 'day_night_app_ratio',
        'high_value_customer', 'heavy_data_user', 'multi_service_user',
        'age_group_encoded', 'tenure_group_encoded', 'arpu_group_encoded'
    ]
    
    feature_columns = base_features + engineered_features
    
    # Handle any remaining infinite or NaN values
    X = data[feature_columns].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    
    if is_training:
        # Fill any remaining NaN values with median for numeric columns
        numeric_columns = X.select_dtypes(include=[np.number]).columns
        X[numeric_columns] = X[numeric_columns].fillna(X[numeric_columns].median())
    else:
        # Use the same median values from training
        numeric_columns = X.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            if col in X.columns:
                X[col] = X[col].fillna(X[col].median())
    
    print(f"Feature matrix shape: {X.shape}")
    print(f"Number of features: {len(feature_columns)}")
    
    if is_training:
        y = data['label']
        fitted_objects = {
            'encoders': encoders,
            'imputers': imputers
        }
        return X, y, feature_columns, fitted_objects
    else:
        return X, feature_columns

# Usage example:
def analyze_dataset(df):
    """
    Quick analysis of the dataset
    """
    print("Dataset Analysis:")
    print(f"Shape: {df.shape}")
    print(f"\nTarget distribution:")
    print(df['label'].value_counts(normalize=True))
    
    print(f"\nMissing values by column:")
    missing_cols = df.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
    print(missing_cols.head(10))
    
    print(f"\nBasic statistics for key features:")
    key_features = ['age', 'innet_dura', 'arpu', 'l3m_avg_mou', 'l3m_avg_dou']
    print(df[key_features].describe())

In [ ]:
# # Run analysis
# analyze_dataset(df)

# # Preprocess the data
# print("\nPreprocessing data...")
# X, y, feature_columns, fitted_objects = preprocess_data(df, is_training=True)

# print(f"\nPreprocessing complete!")
# print(f"Features shape: {X.shape}")
# print(f"Target shape: {y.shape}")
# print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")

# Preprocess the data
X, y, feature_columns, fitted_objects = preprocess_data(df, is_training=True)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target distribution in training set:")
print(y_train.value_counts(normalize=True))

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train multiple models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}")
    print('='*50)
    
    # Use scaled data for Logistic Regression, original for Random Forest
    if name == 'Logistic Regression':
        X_train_model = X_train_scaled
        X_test_model = X_test_scaled
    else:
        X_train_model = X_train
        X_test_model = X_test
    
    # Train the model
    model.fit(X_train_model, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_model)
    y_pred_proba = model.predict_proba(X_test_model)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train_model, y_train, cv=5, scoring='roc_auc')
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Cross-validation ROC AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

# Feature importance for Random Forest
if 'Random Forest' in results:
    print(f"\n{'='*50}")
    print("Feature Importance (Random Forest)")
    print('='*50)
    
    rf_model = results['Random Forest']['model']
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance.head(15))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
    plt.title('Top 15 Feature Importances (Random Forest)')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

# Compare models
print(f"\n{'='*50}")
print("Model Comparison")
print('='*50)

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[name]['accuracy'] for name in results.keys()],
    'Test ROC AUC': [results[name]['roc_auc'] for name in results.keys()],
    'CV ROC AUC (mean)': [results[name]['cv_mean'] for name in results.keys()],
    'CV ROC AUC (std)': [results[name]['cv_std'] for name in results.keys()]
})

print(comparison_df)

# Plot ROC curves
plt.figure(figsize=(10, 8))
for name in results.keys():
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_pred_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Select best model based on ROC AUC
best_model_name = max(results.keys(), key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']

print(f"\nBest model: {best_model_name}")
print(f"Best model ROC AUC: {results[best_model_name]['roc_auc']:.4f}")

# Final model evaluation
print(f"\n{'='*50}")
print(f"Final Model Evaluation: {best_model_name}")
print('='*50)

best_y_pred = results[best_model_name]['y_pred']
best_y_pred_proba = results[best_model_name]['y_pred_proba']

print("Detailed Classification Report:")
print(classification_report(y_test, best_y_pred, target_names=['Negative', 'Positive']))

# Confusion matrix heatmap
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Save the best model (optional)
import joblib
joblib.dump(best_model, f'best_classifier_{best_model_name.lower().replace(" ", "_")}.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

print(f"\nModel saved as: best_classifier_{best_model_name.lower().replace(' ', '_')}.pkl")
print("Scaler saved as: feature_scaler.pkl")

In [ ]:
df_test = pd.read_csv('/home/jovyan/output/data/agentA/testA.csv')

In [ ]:
# Print basic information about the dataset
print("Dataset Shape:")
print(f"Rows: {df_test.shape[0]}, Columns: {df_test.shape[1]}")
print("\n" + "="*50)

# Display first few rows
print("First 5 rows:")
print(df_test.head())
print("\n" + "="*50)

# Display basic info about the dataframe
print("Dataset Info:")
print(df_test.info())
print("\n" + "="*50)

# Display statistical summary
print("Statistical Summary:")
print(df_test.describe())
print("\n" + "="*50)

# Check for missing values
print("Missing Values:")
print(df_test.isnull().sum())
print("\n" + "="*50)

# Display column names and data types
print("Columns and Data Types:")
for col in df_test.columns:
    print(f"{col}: {df_test[col].dtype}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

def train_and_predict_pipeline(df_train, df_test):
    """
    Complete pipeline to train models and predict on test set
    """
    print("="*60)
    print("TELECOM CUSTOMER CLASSIFICATION PIPELINE")
    print("="*60)
    
    # Preprocess training data
    print("\n1. Preprocessing training data...")
    X_train, y_train, feature_columns, fitted_objects = preprocess_data(df_train, is_training=True)
    
    print(f"Training data shape: {X_train.shape}")
    print(f"Target distribution in training:")
    print(y_train.value_counts(normalize=True))
    
    # Preprocess test data
    print("\n2. Preprocessing test data...")
    X_test, _ = preprocess_data(df_test, 
                                encoders=fitted_objects['encoders'],
                                imputers=fitted_objects['imputers'],
                                is_training=False)
    
    print(f"Test data shape: {X_test.shape}")
    
    # Split training data for validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )
    
    # Scale features
    print("\n3. Scaling features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_split)
    X_val_scaled = scaler.transform(X_val_split)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models
    models = {
        'Random Forest': {
            'model': RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ),
            'use_scaled': False
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                random_state=42
            ),
            'use_scaled': False
        },
        'Logistic Regression': {
            'model': LogisticRegression(
                random_state=42,
                max_iter=1000,
                C=1.0
            ),
            'use_scaled': True
        }
    }
    
    print("\n4. Training and evaluating models...")
    results = {}
    
    for name, model_config in models.items():
        print(f"\n{'='*40}")
        print(f"Training {name}")
        print('='*40)
        
        model = model_config['model']
        use_scaled = model_config['use_scaled']
        
        # Select appropriate data
        if use_scaled:
            X_train_model = X_train_scaled
            X_val_model = X_val_scaled
            X_test_model = X_test_scaled
        else:
            X_train_model = X_train_split
            X_val_model = X_val_split
            X_test_model = X_test
        
        # Train model
        model.fit(X_train_model, y_train_split)
        
        # Validation predictions
        y_val_pred = model.predict(X_val_model)
        y_val_pred_proba = model.predict_proba(X_val_model)[:, 1]
        
        # Test predictions
        y_test_pred = model.predict(X_test_model)
        y_test_pred_proba = model.predict_proba(X_test_model)[:, 1]
        
        # Calculate metrics
        val_accuracy = accuracy_score(y_val_split, y_val_pred)
        val_roc_auc = roc_auc_score(y_val_split, y_val_pred_proba)
        
        # Cross-validation on full training data
        if use_scaled:
            X_full_scaled = scaler.fit_transform(X_train)
            cv_scores = cross_val_score(model, X_full_scaled, y_train, cv=5, scoring='roc_auc')
        else:
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
        
        results[name] = {
            'model': model,
            'val_accuracy': val_accuracy,
            'val_roc_auc': val_roc_auc,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'test_predictions': y_test_pred,
            'test_probabilities': y_test_pred_proba,
            'use_scaled': use_scaled
        }
        
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        print(f"Validation ROC AUC: {val_roc_auc:.4f}")
        print(f"CV ROC AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        print("\nValidation Classification Report:")
        print(classification_report(y_val_split, y_val_pred))
    
    # Select best model
    best_model_name = max(results.keys(), key=lambda x: results[x]['cv_mean'])
    best_result = results[best_model_name]
    
    print(f"\n{'='*60}")
    print(f"BEST MODEL: {best_model_name}")
    print(f"Cross-validation ROC AUC: {best_result['cv_mean']:.4f}")
    print("="*60)
    
    # Feature importance for tree-based models
    if best_model_name in ['Random Forest', 'Gradient Boosting']:
        print(f"\nTop 20 Feature Importances ({best_model_name}):")
        feature_importance = pd.DataFrame({
            'feature': feature_columns,
            'importance': best_result['model'].feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(feature_importance.head(20).to_string(index=False))
    
    # Create final predictions
    print(f"\n5. Creating final predictions using {best_model_name}...")
    
    # Retrain on full training data for final predictions
    best_model = models[best_model_name]['model']
    
    if best_result['use_scaled']:
        X_train_final = scaler.fit_transform(X_train)
        X_test_final = scaler.transform(X_test)
    else:
        X_train_final = X_train
        X_test_final = X_test
    
    # Final training
    best_model.fit(X_train_final, y_train)
    
    # Final predictions
    final_predictions = best_model.predict(X_test_final)
    final_probabilities = best_model.predict_proba(X_test_final)[:, 1]
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'id': df_test['id'],
        'label': final_predictions
    })
    
    # Also create version with probabilities for analysis
    results_with_proba = pd.DataFrame({
        'id': df_test['id'],
        'label': final_predictions,
        'probability': final_probabilities
    })
    
    # Print prediction summary
    print(f"\nPrediction Summary:")
    print(f"Total test samples: {len(results_df)}")
    print(f"Predicted positive (label=1): {final_predictions.sum()}")
    print(f"Predicted negative (label=0): {len(final_predictions) - final_predictions.sum()}")
    print(f"Positive rate: {final_predictions.mean():.3f}")
    
    # Model comparison
    print(f"\n{'='*60}")
    print("MODEL COMPARISON")
    print("="*60)
    
    comparison_df = pd.DataFrame({
        'Model': list(results.keys()),
        'Validation Accuracy': [results[name]['val_accuracy'] for name in results.keys()],
        'Validation ROC AUC': [results[name]['val_roc_auc'] for name in results.keys()],
        'CV ROC AUC (mean)': [results[name]['cv_mean'] for name in results.keys()],
        'CV ROC AUC (std)': [results[name]['cv_std'] for name in results.keys()]
    })
    
    print(comparison_df.to_string(index=False))
    
    return results_df, results_with_proba, best_model, best_model_name, results

# Execute the complete pipeline
print("Starting prediction pipeline...")

# Assuming df is your training data and df_test is your test data
df_train = df  # Your training data (59904 rows with 'label' column)
# df_test is already defined from your input (19999 rows without 'label' column)

# Run the complete pipeline
results_df, results_with_proba, best_model, model_name, all_results = train_and_predict_pipeline(df_train, df_test)

# Save results
print(f"\n6. Saving results...")

# Save main results (competition format)
results_df.to_csv('test_predictions.csv', index=False)
print(f"✓ Main predictions saved to: test_predictions.csv")

# Save detailed results with probabilities
results_with_proba.to_csv('test_predictions_with_probabilities.csv', index=False)
print(f"✓ Detailed predictions saved to: test_predictions_with_probabilities.csv")

# Save model performance summary
model_performance = pd.DataFrame({
    'Model': list(all_results.keys()),
    'Validation_Accuracy': [all_results[name]['val_accuracy'] for name in all_results.keys()],
    'Validation_ROC_AUC': [all_results[name]['val_roc_auc'] for name in all_results.keys()],
    'CV_ROC_AUC_Mean': [all_results[name]['cv_mean'] for name in all_results.keys()],
    'CV_ROC_AUC_Std': [all_results[name]['cv_std'] for name in all_results.keys()]
})
model_performance.to_csv('model_performance_summary.csv', index=False)
print(f"✓ Model performance summary saved to: model_performance_summary.csv")

# Display final results
print(f"\n{'='*60}")
print("FINAL RESULTS")
print("="*60)
print(f"Best model: {model_name}")
print(f"Test predictions saved to: test_predictions.csv")
print(f"Format: id, label (0 or 1)")
print(f"\nFirst 10 predictions:")
print(results_df.head(10).to_string(index=False))

print(f"\nPrediction distribution:")
print(results_df['label'].value_counts().to_string())
print(f"\nPipeline completed successfully! ✓")